# 02. The Data Customs Officer: Schema-Gated Agents
**Design Pattern:** The 4-Gate Inspection Lane  
**LLM:** `openai/gpt-oss-120b` via Groq Cloud  
**Validation Engine:** Pydantic v2  

The **Data Customs Officer** pattern treats natural language LLM completions as unverified international cargo. Before any payload touches internal application memory, databases, or microservice buses, it must pass through a strict, 4-gate verification boundary.

In [1]:
!pip install -q groq pydantic

import os
import json
from datetime import datetime, date
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, field_validator
from groq import Groq

# Configure Groq Client
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL_ID = "openai/gpt-oss-120b"
print("Client initialized successfully.")

Client initialized successfully.


## 1. The 4-Gate Inspection Lane Contract
```mermaid
graph LR
    G1["Gate 1: Input<br>*(Raw Prompt Text)*"]
    G2["Gate 2: Schema Contract<br>*(Pydantic Type Guard)*"]
    G3["Gate 3: Agent Extraction<br>*(Tool-Calling Enforcement)*"]
    G4["Gate 4: Validated Record<br>*(Typed Safe Object)*"]

    G1 --> G2 --> G3 --> G4
    
    style G1 fill:#d9d9d9,stroke:#1a1a1a,stroke-width:2px,color:#000000
    style G2 fill:#b3d9ff,stroke:#1a1a1a,stroke-width:2px,color:#000000
    style G3 fill:#ffd9b3,stroke:#1a1a1a,stroke-width:2px,color:#000000
    style G4 fill:#b3ffb3,stroke:#1a1a1a,stroke-width:2px,color:#000000
```

In [2]:
# Gate 2: Schema Definition
# Hardened Pydantic Contract with Real Date Objects & Logic Checks
class FlightBooking(BaseModel):
    intent: Literal["flight_booking"] = Field(
        default="flight_booking",
        description="The primary intent classification"
    )
    passenger_name: str = Field(description="Full name of the passenger")
    origin: str = Field(description="Departure city or airport origin code")
    destination: str = Field(description="Destination city or airport arrival code")
    raw_date_expression: str = Field(
        description="Verbatim date phrase spoken by user, e.g. 'next Friday'"
    )
    departure_date: date = Field(
        description="Exact computed ISO-8601 departure date formatted as YYYY-MM-DD"
    )
    seat_preference: Literal["aisle_quiet", "aisle", "window", "middle"] = Field(
        description="Seat placement selection"
    )
    budget_limit_usd: int = Field(description="Maximum numeric budget limit in USD")
    meal: Literal["vegan", "standard", "none"] = Field(
        description="Standardized catering category selection"
    )

    @field_validator("departure_date")
    @classmethod
    def verify_not_in_past(cls, v: date) -> date:
        if v < date.today():
            raise ValueError(f"Departure date {v} cannot be in the past.")
        return v

# Generate standard JSON Schema for Groq tool calling
flight_schema = {
    "name": "submit_flight_booking",
    "description": "Records and validates a clean flight booking payload with exact origin, destination, and calendar dates.",
    "parameters": FlightBooking.model_json_schema()
}

print("JSON Schema Contract:")
print(json.dumps(flight_schema, indent=2))


JSON Schema Contract:
{
  "name": "submit_flight_booking",
  "description": "Records and validates a clean flight booking payload with exact origin, destination, and calendar dates.",
  "parameters": {
    "properties": {
      "intent": {
        "const": "flight_booking",
        "default": "flight_booking",
        "description": "The primary intent classification",
        "title": "Intent",
        "type": "string"
      },
      "passenger_name": {
        "description": "Full name of the passenger",
        "title": "Passenger Name",
        "type": "string"
      },
      "origin": {
        "description": "Departure city or airport origin code",
        "title": "Origin",
        "type": "string"
      },
      "destination": {
        "description": "Destination city or airport arrival code",
        "title": "Destination",
        "type": "string"
      },
      "raw_date_expression": {
        "description": "Verbatim date phrase spoken by user, e.g. 'next Friday'",
       

In [3]:
# Gate 1 & Gate 3: Agent Execution via Tool-Calling
user_input = "Book me a flight from San Francisco to Tokyo next Friday under 900 bucks, make sure it's an aisle seat and a vegan meal. Name is Jaichand."
def run_customs_officer(raw_user_text: str) -> str:
    """
    Executes model extraction by grounding the system clock dynamically at runtime.
    """
    now = datetime.now()
    anchor_date = now.strftime("%Y-%m-%d")
    anchor_weekday = now.strftime("%A")

    system_instruction = f"""
You are an expert flight reservation customs extraction officer.
TEMPORAL GROUND TRUTH ANCHOR:
- Today's date is: {anchor_date} ({anchor_weekday}).
- All relative dates (e.g. 'next Friday', 'tomorrow', 'next week') MUST be strictly calculated starting from this reference anchor.
- 'next Friday' refers to the immediate upcoming Friday strictly after today ({anchor_weekday}, {anchor_date}).
- Return the exact ISO-8601 date string (YYYY-MM-DD) for 'departure_date'.
- Extract both the departure origin and arrival destination.
Invoke the submit_flight_booking function with your extracted arguments.
"""

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": raw_user_text}
        ],
        tools=[{"type": "function", "function": flight_schema}],
        tool_choice={"type": "function", "function": {"name": "submit_flight_booking"}},
        temperature=0.0
    )
    
    return response.choices[0].message.tool_calls[0].function.arguments

# Alias so both function names work seamlessly
run_groq_customs_officer = run_customs_officer

print("✓ Extraction function `run_customs_officer` ready.")

print("Raw Extracted Arguments from Tool Call:")
extracted_json_str = run_customs_officer(user_input)
print(extracted_json_str)

print(f"[System Clock] Current Date: {datetime.now().strftime('%A, %Y-%m-%d')}")
raw_tool_args = run_groq_customs_officer(user_input)

✓ Extraction function `run_customs_officer` ready.
Raw Extracted Arguments from Tool Call:
{"budget_limit_usd":900,"departure_date":"2026-09-25","destination":"Tokyo","meal":"vegan","origin":"San Francisco","passenger_name":"Jaichand","raw_date_expression":"next Friday","seat_preference":"aisle"}
[System Clock] Current Date: Wednesday, 2026-09-23


In [4]:
# Gate 4: Runtime Enforcement via Pydantic & Business Logic Verification
try:
    # 1. Schema Validation and Deserialization
    validated_booking = FlightBooking.model_validate_json(raw_tool_args)
    
    booking_date = validated_booking.departure_date
    actual_weekday = booking_date.strftime("%A")

    print("\n================ PASSED GROQ CUSTOMS CLEARANCE ================")
    print(f"Passenger Name  : {validated_booking.passenger_name}")
    print(f"Origin (From)   : {validated_booking.origin}")
    print(f"Destination (To): {validated_booking.destination}")
    print(f"User Request    : '{validated_booking.raw_date_expression}'")
    print(f"Departure Date  : {booking_date} (Confirmed: {actual_weekday})")
    print(f"Seat Preference : {validated_booking.seat_preference}")
    print(f"Meal Choice     : {validated_booking.meal}")
    print(f"Budget Limit    : ${validated_booking.budget_limit_usd:,} USD")
    print("===============================================================")

    # 2. Dynamic Calendar Integrity Assertion
    # Verifies whatever weekday was mentioned in the user expression matches the resolved date
    weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    user_phrase = validated_booking.raw_date_expression.lower()
    
    for day in weekdays:
        if day.lower() in user_phrase and actual_weekday != day:
            raise ValueError(
                f"Calendar Logic Failure: User requested '{day}', but calendar resolved to {actual_weekday} ({booking_date})!"
            )

    # 3. Clean JSON Output (Safe for APIs / SQL)
    cleared_json_payload = validated_booking.model_dump_json(indent=2)
    print("\n[Inspected Payload - Ready for Database/Stripe]:")
    print(cleared_json_payload)

    print(f"\n Ticket safely cleared. Flight confirmed from {validated_booking.origin} to {validated_booking.destination} on {actual_weekday} ({booking_date}).")

except ValidationError as val_err:
    print(f"\n❌ REJECTED AT GATE 4 (Pydantic Schema Error):\n{val_err}")
except ValueError as logic_err:
    print(f"\n❌ REJECTED AT GATE 4 (Business / Calendar Logic Failure):\n{logic_err}")


================ PASSED GROQ CUSTOMS CLEARANCE ================
Passenger Name  : Jaichand
Origin (From)   : San Francisco
Destination (To): Tokyo
User Request    : 'next Friday'
Departure Date  : 2026-09-25 (Confirmed: Friday)
Seat Preference : aisle
Meal Choice     : vegan
Budget Limit    : $900 USD

[Inspected Payload - Ready for Database/Stripe]:
{
  "intent": "flight_booking",
  "passenger_name": "Jaichand",
  "origin": "San Francisco",
  "destination": "Tokyo",
  "raw_date_expression": "next Friday",
  "departure_date": "2026-09-25",
  "seat_preference": "aisle",
  "budget_limit_usd": 900,
  "meal": "vegan"
}

 Ticket safely cleared. Flight confirmed from San Francisco to Tokyo on Friday (2026-09-25).


## 2. Interactive Terminal: The Live Customs Inspection Lane

Run the widget below to inspect payloads, visualize real-time pipeline status, and contrast unstructured extraction against schema-enforced extraction.

In [5]:
import time
import traceback
from IPython.display import display, HTML, clear_output

# Updated test payload containing both Origin and Destination
user_input_text = "Book me a flight from San Francisco to Tokyo next Friday under 900 bucks, make sure it's an aisle seat and a vegan meal. Name is Jaichand."

def render_dashboard(g1="idle", g2="idle", g3="idle", g4="idle", left_msg="Waiting...", right_msg="Waiting...", status="READY"):
    gate_styles = {
        "idle": "background:#EDE6DA; color:#6E6254;",
        "active": "background:#E8A838; color:#FFFFFF; font-weight:bold; box-shadow: 0 0 10px rgba(232,168,56,0.6);",
        "passed": "background:#15803D; color:#FFFFFF; font-weight:bold;",
        "failed": "background:#DC2626; color:#FFFFFF; font-weight:bold;"
    }
    
    html = f"""
    <div style="background-color: #FBF9F5; border: 1px solid #E2D9CC; border-radius: 12px; padding: 22px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 940px; box-shadow: 0 4px 14px rgba(0,0,0,0.04);">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px;">
            <div>
                <h3 style="margin: 0; font-size: 18px; color: #2D2823;">Flight Data Customs Terminal</h3>
                <small style="color: #786C5E;">Host Clock: {datetime.now().strftime('%A, %Y-%m-%d')}</small>
            </div>
            <span style="background: #EDE6DA; color: #374151; padding: 5px 14px; border-radius: 12px; font-size: 11px; font-weight: bold; letter-spacing: 0.5px;">
                {status}
            </span>
        </div>

        <!-- 4-Gate Pipeline Track -->
        <div style="display: flex; justify-content: space-between; gap: 8px; margin-bottom: 18px;">
            <div style="flex:1; text-align:center; padding:10px 4px; border-radius:8px; font-size:12px; {gate_styles[g1]}">Gate 1: Input</div>
            <div style="flex:1; text-align:center; padding:10px 4px; border-radius:8px; font-size:12px; {gate_styles[g2]}">Gate 2: Schema</div>
            <div style="flex:1; text-align:center; padding:10px 4px; border-radius:8px; font-size:12px; {gate_styles[g3]}">Gate 3: Agent</div>
            <div style="flex:1; text-align:center; padding:10px 4px; border-radius:8px; font-size:12px; {gate_styles[g4]}">Gate 4: Validated</div>
        </div>

        <!-- Two Comparison Panels -->
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px;">
            <div style="background: #FDF2F2; border: 1px solid #F8B4B4; border-radius: 8px; padding: 14px; min-height: 260px; overflow-x: auto;">
                <div style="color: #9B1C1C; font-weight: bold; font-size: 12px; margin-bottom: 8px;">Plain LLM Response (Unvalidated Cargo)</div>
                <pre style="margin:0; font-family: monospace; font-size: 11.5px; white-space: pre-wrap; word-break: break-all; color: #9B1C1C;">{left_msg}</pre>
            </div>
            <div style="background: #F0FDF4; border: 1px solid #BBF7D0; border-radius: 8px; padding: 14px; min-height: 260px; overflow-x: auto;">
                <div style="color: #166534; font-weight: bold; font-size: 12px; margin-bottom: 8px;">Structured Output (Inspected & Cleared)</div>
                <pre style="margin:0; font-family: monospace; font-size: 11.5px; white-space: pre-wrap; word-break: break-all; color: #166534;">{right_msg}</pre>
            </div>
        </div>
    </div>
    """
    clear_output(wait=True)
    display(HTML(html))

# --- STEP-BY-STEP INSPECTION RUNNER ---

# Step 1: Ingesting text
render_dashboard(
    g1="active", g2="idle", g3="idle", g4="idle",
    left_msg="Querying unconstrained LLM completion...",
    right_msg="Awaiting clearance initiation...",
    status="GATE 1: INGESTION"
)

try:
    plain_resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": f"Extract flight details to JSON: {user_input_text}"}],
        temperature=0.7
    ).choices[0].message.content or "Empty output"
except Exception as e:
    plain_resp = f"API Error in Gate 1:\n{e}"

# Step 2: Schema Locking
render_dashboard(
    g1="passed", g2="active", g3="idle", g4="idle",
    left_msg=plain_resp,
    right_msg="Locking Pydantic JSON Schema Contract with Origin & ISO Date verification...",
    status="GATE 2: SCHEMA LOCK"
)
time.sleep(0.5)

# Step 3: Agent Extraction via Tool-calling
render_dashboard(
    g1="passed", g2="passed", g3="active", g4="idle",
    left_msg=plain_resp,
    right_msg="Sending clock-grounded tool call to model...",
    status="GATE 3: AGENT EXTRACTION"
)

try:
    raw_args = run_customs_officer(user_input_text)
    
    # Step 4: Verification & Hydration
    render_dashboard(
        g1="passed", g2="passed", g3="passed", g4="active",
        left_msg=plain_resp,
        right_msg=f"Raw payload returned:\n{raw_args}\n\nRunning Pydantic validation & calendar assertion...",
        status="GATE 4: VERIFYING"
    )
    time.sleep(0.4)
    
    # Deserialization and validation
    validated = FlightBooking.model_validate_json(raw_args)
    
    # Business Logic Verification: Weekday alignment check
    booking_date = validated.departure_date
    actual_weekday = booking_date.strftime("%A")
    weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    user_phrase = validated.raw_date_expression.lower()
    
    for day in weekdays:
        if day.lower() in user_phrase and actual_weekday != day:
            raise ValueError(
                f"Calendar Logic Failure: User requested '{day}', but calendar resolved to {actual_weekday} ({booking_date})!"
            )

    # Native serialization safe for date objects
    cleared_json = validated.model_dump_json(indent=2)
    
    # All Gates Passed
    render_dashboard(
        g1="passed", g2="passed", g3="passed", g4="passed",
        left_msg=plain_resp,
        right_msg=cleared_json,
        status="PASSED CUSTOMS CLEARANCE"
    )

except ValidationError as val_err:
    render_dashboard(
        g1="passed", g2="passed", g3="passed", g4="failed",
        left_msg=plain_resp,
        right_msg=f"IMPOUNDED AT GATE 4 (Schema Violation):\n{val_err}",
        status="REJECTED BY SCHEMA"
    )
except ValueError as logic_err:
    render_dashboard(
        g1="passed", g2="passed", g3="passed", g4="failed",
        left_msg=plain_resp,
        right_msg=f"IMPOUNDED AT GATE 4 (Calendar Check):\n{logic_err}",
        status="REJECTED BY CALENDAR"
    )
except Exception as general_err:
    render_dashboard(
        g1="passed", g2="passed", g3="failed", g4="failed",
        left_msg=plain_resp,
        right_msg=f"CRITICAL ERROR:\n{general_err}\n\n{traceback.format_exc()}",
        status="ERROR"
    )

> **Note:** The comparison below reflects a real sample test run. Because unconstrained LLMs are non-deterministic, **your exact keys and hallucinated values may vary across runs**. However, the failure patterns (type drift, key mutation, missing fields) remain consistent without schema gating.

---

### Empirical Observation: Unstructured Variance vs. Structured Output

| Feature / Field | Unstructured Run 1 | Unstructured Run 2 | Unstructured Run 3 | Structured Output |
| --- | --- | --- | --- | --- |
| **Origin Type** | `dict` (`{"city": "...", "airport_code": "..."}`) | `str` (`"San Francisco (SFO)"`) | `str` (`"San Francisco (SFO)"`) | `str` (`"San Francisco"`) |
| **Destination Value** | Nested object with airport code (`NRT`) | Appended note: `"Tokyo (any airport)"` | Appended note: `"Tokyo (any airport)"` | Clean value: `"Tokyo"` |
| **Date Key Name** | `"departure_date"` | `"departure_date"` | Renamed: `"date"` | Guaranteed: `"departure_date"` |
| **Computed Date** | `2026-09-27` (Sunday) | `2026-09-26` (Saturday) | `2026-09-27` (Sunday) | `2026-09-25` (Friday) |
| **Budget Key Name** | Renamed: `"budget_usd"` | Renamed: `"budget_usd"` | Renamed: `"budget_usd"` | Guaranteed: `"budget_limit_usd"` |
| **Meal Key Name** | Renamed: `"meal_preference"` | Renamed: `"meal_preference"` | Renamed: `"meal_preference"` | Guaranteed: `"meal"` |
| **Key Ordering** | Standard top-to-bottom | Standard top-to-bottom | Scrambled (`origin` first, `passenger_name` last) | Consistent canonical schema order |
| **Intent Tag** | Missing | Missing | Missing | Included: `"intent": "flight_booking"` |
